# 第2章 GPU 体系结构（上）：编程模型与波前执行 - 操作手册

**Goal**: 理解 GPU 线程层级（grid/workgroup/wavefront/lane）、波前执行模型和分支发散现象。

**Prerequisite**: 已完成第1章环境验证，ROCm 和 HIP 编译工具链可用。

**Platform**: 原生 Ubuntu 24.04（推荐）或 WSL2，gfx1201 为叙述基线。

**参考文档**: `docs/part0-intro/chapter2/index.md`

**代码目录**: `code/part0-intro/chapter2/`

## 1. 定位仓库根目录

## 2. 检测 GPU 架构

**Parameter**: 无

**Execution**: 运行 `rocminfo` 检测当前 GPU 架构。

**Expected output**: 输出检测到的架构（gfx1100/gfx1151/gfx1201）。

**Pass criteria**: 成功检测到支持的架构之一。

In [ ]:
# 检测当前 GPU 架构
import subprocess

rocminfo_result = subprocess.run(
    ["rocminfo"],
    capture_output=True,
    text=True,
    check=True
)

# 从 rocminfo 输出中提取架构
arch = "gfx1201"  # 默认值（gfx1201 为叙述基线）
if "gfx1100" in rocminfo_result.stdout:
    arch = "gfx1100"
elif "gfx1151" in rocminfo_result.stdout:
    arch = "gfx1151"
elif "gfx1201" in rocminfo_result.stdout:
    arch = "gfx1201"
else:
    raise RuntimeError("未检测到支持的架构 (gfx1100/gfx1151/gfx1201)")

print(f"检测到架构: {arch}")


In [ ]:
import pathlib
import subprocess

def find_repo_root():
    current = pathlib.Path.cwd().resolve()
    for candidate in [current] + list(current.parents):
        if (
            (candidate / "README.md").exists()
            and (candidate / "code").is_dir()
            and (candidate / "docs").is_dir()
            and (candidate / "notebooks").is_dir()
        ):
            return candidate
    raise FileNotFoundError("无法定位仓库根目录")

REPO_ROOT = find_repo_root()
print(f"仓库根目录: {REPO_ROOT}")

## 3. 线程层级映射表（N=10, block=4）

**Parameter**: 
- N=10（总线程数）
- block=4（每个 workgroup 的线程数）

**Execution**: 生成 thread mapping 表，展示 grid → workgroup → wavefront → lane 的映射关系。

**Expected output**: 表格显示每个线程的 blockIdx、threadIdx、全局 tid 和所属 wavefront。

**Pass criteria**: 表格正确显示 N=10 个线程的完整映射关系。

In [ ]:
N = 10
block = 4
wave_size = 32  # gfx1201 使用 wave32

# 生成 thread mapping 表
data = []
for tid in range(N):
    blockIdx = tid // block
    threadIdx = tid % block
    wavefront = threadIdx // wave_size
    lane = threadIdx % wave_size
    data.append({
        "tid": tid,
        "blockIdx.x": blockIdx,
        "threadIdx.x": threadIdx,
        "wavefront": wavefront,
        "lane": lane
    })

print("\nThread Mapping Table (N=10, block=4, wave32):")
headers = ("tid", "blockIdx.x", "threadIdx.x", "wavefront", "lane")
print(" ".join(f"{header:>12}" for header in headers))
for row in data:
    print(" ".join(f"{row[header]:>12}" for header in headers))

# 边界 lanes 说明
print("\n边界说明:")
print(f"- 总线程数 N={N}")
print(f"- 每个 workgroup 包含 {block} 个线程")
print(f"- 需要 {(N + block - 1) // block} 个 workgroups")
print(f"- 最后一个 workgroup 只有 {N % block if N % block != 0 else block} 个有效线程")
print(f"- 每个 wavefront 最多 {wave_size} 个 lanes（gfx1201 使用 wave32）")

## 4. 分支发散实验

**Parameter**:
- 实现: wave-uniform（无分支发散）vs wave-divergent（有分支发散）
- 数据规模: 16M 元素
- warmup: 5 次
- repeat: 20 次

**Execution**: 编译并运行 `branch_divergence.hip`，对比两种实现的性能。

**Expected output**: 
- wave-uniform 实现的执行时间
- wave-divergent 实现的执行时间
- 性能差异百分比

**Pass criteria**: 程序成功编译运行，输出包含两种实现的性能数据。

**Platform-specific commands**:
- gfx1100: `hipcc --offload-arch=gfx1100 -O3 -std=c++17 branch_divergence.hip -o branch_divergence`
- gfx1151: `hipcc --offload-arch=gfx1151 -O3 -std=c++17 branch_divergence.hip -o branch_divergence`
- gfx1201: `hipcc --offload-arch=gfx1201 -O3 -std=c++17 branch_divergence.hip -o branch_divergence`

In [ ]:
chapter2_dir = REPO_ROOT / "code/part0-intro/chapter2"
branch_div_hip = chapter2_dir / "branch_divergence.hip"
branch_div_bin = chapter2_dir / "branch_divergence"

# 编译（以 gfx1201 为基线）
compile_result = subprocess.run(
    ["hipcc", f"--offload-arch={arch}", "-O3", "-std=c++17", 
     str(branch_div_hip), "-o", str(branch_div_bin)],
    capture_output=True,
    text=True,
    check=True,
    cwd=chapter2_dir
)
print("编译成功")

# 运行实验
run_result = subprocess.run(
    [str(branch_div_bin), "--implementation", "all", "--size", "16777216", 
     "--warmup", "5", "--repeat", "20"],
    capture_output=True,
    text=True,
    check=True,
    cwd=chapter2_dir
)
print("\n实验结果:")
print(run_result.stdout)

print("\n解释:")
print("- wave-uniform: 同一 wavefront 内所有 lanes 执行相同分支，无发散")
print("- wave-divergent: 同一 wavefront 内 lanes 执行不同分支，产生发散")
print("- 分支发散会导致性能下降，因为硬件需要串行执行不同分支")

## 5. 体系结构图示

本节展示 GPU 体系结构的关键概念图示（如果图片文件存在）。

In [ ]:
from IPython.display import Image, display

# 检查并显示图片（如果存在）
images_dir = REPO_ROOT / "docs/part0-intro/chapter2/images"
if images_dir.exists():
    for img_file in images_dir.glob("*.png"):
        print(f"\n图示: {img_file.name}")
        display(Image(filename=str(img_file)))
else:
    print("图片目录不存在，请参考文档中的 Mermaid 图示")

## 总结

本章完成了 GPU 编程模型的核心概念验证：
1. ✓ 理解线程层级映射（grid → workgroup → wavefront → lane）
2. ✓ 观察分支发散对性能的影响
3. ✓ 建立波前执行模型的直觉

这些概念是后续优化工作的基础。